# Cascaded-SolarNet: test + train on Colab

Clones the repo, runs the test suite, downloads InfraredSolarModules, trains the cascade and baselines, and evaluates end to end.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Clone and install

In [ ]:
%cd /content
!rm -rf cascaded-cnn
!git clone https://github.com/techwack/cascaded-cnn
%cd /content/cascaded-cnn
# torch is already installed on Colab; install the rest
!pip install -q numpy scikit-learn pillow pyyaml pytest
!pip install -q -e . --no-deps

## 2. Run the tests (synthetic data, no download needed)

In [ ]:
!python -m pytest -q

## 3. Download the dataset
Fetches InfraredSolarModules from the original RaptorMaps GitHub repo (CC BY 4.0). No Kaggle token needed.

In [ ]:
!bash scripts/download_data.sh
!python scripts/make_splits.py

## 4. Train: two configs, same data and split
- **default**: with brightness/contrast/noise augmentation -> `runs/`
- **no_photometric**: geometric flips only -> `runs_nophoto/`

Each takes a few minutes on a T4. Early stopping is on val macro-F1 (patience 20, up to 100 epochs).

In [ ]:
%cd /content/cascaded-cnn
!python -m solarnet.train --node all
!python -m solarnet.train --node flat

In [ ]:
%cd /content/cascaded-cnn
!python -m solarnet.train --node all --config configs/no_photometric.yaml
!python -m solarnet.train --node flat --config configs/no_photometric.yaml

In [ ]:
%cd /content/cascaded-cnn
!python baselines/random_forest.py

## 5. Evaluate the whole cascade on the test split

In [ ]:
%cd /content/cascaded-cnn
!python -m solarnet.evaluate --flat
!python -m solarnet.evaluate --flat --config configs/no_photometric.yaml

In [ ]:
%cd /content/cascaded-cnn
import json
for name, d in [("default", "runs"), ("no_photometric", "runs_nophoto")]:
    r = json.load(open(f"{d}/results.json"))
    print(f"== {name}")
    for m in ("cascade", "flat"):
        x = r[m]
        print(f"  {m:8s} acc {x['accuracy']:.3f}  macro-F1 {x['macro_f1']:.3f}  kappa {x['kappa']:.3f}  params {x['params']:,}  {x['latency_ms_per_image']:.2f} ms")
    print("  stage errors:", r["cascade"]["stages"])
print("random forest:", json.load(open("runs/random_forest.json")))

## 6. Save results
Downloads logs and metrics (not the large checkpoints).

In [ ]:
%cd /content/cascaded-cnn
!zip -qr results.zip runs runs_nophoto -x "*.pt"
from google.colab import files
files.download("results.zip")